# 📊 네이버 증권 종목토론방 크롤러 + 감성분석 + ETF 채널 분석 v7

특정 날짜 하루치 게시글을 수집하고 **감성분석·키워드·시간대 추이·인사이트**를 분석합니다.
ETF 순매수 데이터를 연계하면 **은행·증권사 채널별 마케팅 인사이트**까지 도출합니다.

## 실행 순서
| 셀 | 내용 |
|---|---|
| ⚙️ 설정 | `STOCK_CODE`, `TARGET_DATE` 입력 |
| 📦 패키지 설치 | openpyxl 설치 (최초 1회) |
| 🔧 함수 정의 | 크롤링 + 감성분석 함수 로드 |
| 🔍 수집 | 게시글 목록 + 본문 크롤링 |
| 🧠 감성분석 | 한국어 금융 감성사전 기반 분석 |
| 🔑 키워드·추이 | 빈도 키워드 + 시간대별 게시글 수 |
| 📂 ETF 파일 업로드 | ETF 순매수 엑셀 업로드 (선택) |
| 📊 ETF 채널 분석 | 은행·증권사 채널 수급 분석 |
| 💡 인사이트 | 종토방 심리 + ETF 채널 통합 리포트 |
| 💾 엑셀 저장 | 6개 시트 엑셀 다운로드 |

> **ETF 분석은 선택 사항**입니다. 파일 없이도 종토방 인사이트만 단독 실행 가능합니다.


In [ ]:
# ===================== ⚙️ 설정 (여기만 바꾸면 됩니다) =====================
STOCK_CODE  = "005930"      # 종목코드. 예) 000660 SK하이닉스 / 035420 NAVER
TARGET_DATE = "2026.06.08"  # 수집할 날짜 (YYYY.MM.DD)
DELAY_SEC   = 1.2           # 요청 간 대기(초)
# =========================================================================
print(f"✅ 설정 완료 — 종목: {STOCK_CODE} / 날짜: {TARGET_DATE}")

In [ ]:
!pip install -q -U openpyxl
print("✅ 패키지 설치 완료")

In [ ]:
import requests, re, time, json
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs
from datetime import datetime
from collections import Counter
from tqdm.auto import tqdm

BASE      = "https://finance.naver.com"
FRONT_API = "https://m.stock.naver.com/front-api"

HEADERS_PC = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Referer": f"{BASE}/item/board.naver?code={STOCK_CODE}",
    "Accept-Language": "ko-KR,ko;q=0.9",
}
HEADERS_API = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Referer": f"https://m.stock.naver.com/pc/domestic/stock/{STOCK_CODE}/discussion/0",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ko-KR,ko;q=0.9",
}

# ══════════════════════════════════════════════════════════════════════════
# 한국어 금융 감성사전
# ══════════════════════════════════════════════════════════════════════════
POS_WORDS = {
    # 상승·수익 관련
    "상승","급등","강세","반등","돌파","신고가","최고가","상한가","매수","호재",
    "기대","긍정","좋다","좋은","좋을","훌륭","우수","성장","개선","회복","확대",
    "증가","상향","올라","오른","오를","뛰어","뚫","넘","목표","달성","성공",
    "흑자","수익","이익","배당","실적","호실적","어닝","서프라이즈","깜짝",
    "저평가","매력","유망","기회","기대감","긍정적","낙관","상승세","우상향",
    "강력","탁월","완벽","최강","압도","독보","선도","혁신","첨단","경쟁력",
    "안정","견조","탄탄","건강","활발","활성","활황","붐","랠리","재평가",
    "모아","줍줍","불타기","존버","장투","믿음","확신","잠재","가능성",
    "저가","저점","바닥","반전","턴어라운드","구조조정","체질개선","정상화",
}
NEG_WORDS = {
    # 하락·손실 관련
    "하락","급락","약세","폭락","추락","신저가","최저가","하한가","매도","악재",
    "우려","부정","나쁘","나쁜","나쁠","최악","실망","위기","악화","축소",
    "감소","하향","내려","떨어","떨어질","빠져","뚫리","무너","목표하향","실패",
    "적자","손실","손해","부채","위험","리스크","불안","불확실","불투명",
    "고평가","거품","버블","과열","조정","횡보","정체","침체","둔화","부진",
    "실적악화","어닝쇼크","쇼크","충격","경고","주의","위험","파산","청산",
    "손절","탈출","도망","포기","공매도","숏","하방","추가하락","더빠질",
    "불안","걱정","두렵","무섭","겁","허탈","억울","화난","짜증","분노",
    "속임","사기","작전","세력","개미","털리","당했","물렸","물려","깊은물",
    "반토막","반절","회의","의심","신뢰잃","불신","망할","망하","끝났",
}
NEG_INTENSIFIERS = {"매우","너무","완전","진짜","정말","엄청","극도로","심각","최악의"}
POS_INTENSIFIERS = {"매우","너무","완전","진짜","정말","엄청","극도로","대단히","최고의"}

def _score_text(text: str) -> tuple[int, list, list]:
    """
    텍스트에서 긍정·부정 단어 수를 세어 감성 점수 반환.
    반환: (score 1~5, 긍정단어목록, 부정단어목록)
    """
    text = str(text)
    words = re.findall(r"[가-힣]+", text)
    pos_found, neg_found = [], []

    for i, w in enumerate(words):
        # 단어가 감성 사전에 있는지 확인 (부분 매칭 포함)
        for pw in POS_WORDS:
            if pw in w or w in pw:
                # 앞 단어가 부정어인지 확인 ("안 좋다" → 부정 처리)
                prev = words[i-1] if i > 0 else ""
                if prev in {"안","못","절대","전혀","별로","아니","없"}:
                    neg_found.append(w)
                else:
                    pos_found.append(pw)
                break
        for nw in NEG_WORDS:
            if nw in w or w in nw:
                neg_found.append(nw)
                break

    pos_cnt = len(pos_found)
    neg_cnt = len(neg_found)
    diff    = pos_cnt - neg_cnt

    if diff >= 3:   score = 5
    elif diff >= 1: score = 4
    elif diff == 0 and pos_cnt == 0 and neg_cnt == 0: score = 3
    elif diff == 0: score = 3
    elif diff >= -2: score = 2
    else:           score = 1

    return score, list(set(pos_found)), list(set(neg_found))

def analyze_sentiment(text: str) -> dict:
    """단일 텍스트 감성분석. 반환: {sentiment, score, reason}"""
    score, pos_w, neg_w = _score_text(text)
    if score >= 4:
        sentiment = "긍정"
        reason    = f"긍정 표현: {', '.join(pos_w[:3])}" if pos_w else "전반적 긍정 어조"
    elif score <= 2:
        sentiment = "부정"
        reason    = f"부정 표현: {', '.join(neg_w[:3])}" if neg_w else "전반적 부정 어조"
    else:
        sentiment = "중립"
        reason    = "중립적 표현"
    return {"sentiment": sentiment, "score": score, "reason": reason}

# ══════════════════════════════════════════════════════════════════════════
# 크롤링 함수
# ══════════════════════════════════════════════════════════════════════════
def _get_soup(url):
    res = requests.get(url, headers=HEADERS_PC, timeout=15)
    return BeautifulSoup(res.content, "html.parser")

def _parse_date(text):
    text = text.strip()
    for fmt in ("%Y.%m.%d %H:%M", "%Y.%m.%d"):
        try:
            return datetime.strptime(text, fmt).date()
        except ValueError:
            pass
    return None

def get_total_pages(code):
    last, page, visited = 1, 1, set()
    while True:
        if page in visited:
            break
        visited.add(page)
        soup = _get_soup(f"{BASE}/item/board.naver?code={code}&page={page}")
        nums = [
            int(m)
            for a in soup.select("a[href*='board.naver'][href*='page=']")
            for m in re.findall(r"page=(\d+)", a.get("href", ""))
        ]
        if nums:
            last = max(last, max(nums))
        next_a = soup.select_one("td.pgR > a")
        if not next_a:
            break
        m = re.search(r"page=(\d+)", next_a.get("href", ""))
        if not m:
            break
        next_page = int(m.group(1))
        if next_page <= page:
            break
        page = next_page
        time.sleep(0.5)
    return last

def collect_posts_for_date(code, target_date_str, delay=1.2):
    target_date = datetime.strptime(target_date_str, "%Y.%m.%d").date()
    total_pages = get_total_pages(code)
    print(f"종목 {code} 종토방: 전체 {total_pages:,}페이지 확인")
    print(f"▶ {target_date_str} 날짜 게시글 수집 시작\n")

    posts, stop = [], False
    for page in range(1, total_pages + 1):
        if stop:
            break
        soup  = _get_soup(f"{BASE}/item/board.naver?code={code}&page={page}")
        table = soup.select_one("table.type2")
        rows  = table.select("tr") if table else soup.select("tr")
        page_count = 0
        for tr in rows:
            title_td = tr.select_one("td.title")
            if not title_td:
                continue
            a_tag = title_td.select_one("a")
            if not a_tag or not a_tag.get("href"):
                continue
            tds       = tr.find_all("td")
            date_text = tds[0].get_text(strip=True) if tds else ""
            if not date_text:
                continue
            post_date = _parse_date(date_text)
            if post_date is None:
                continue
            if post_date > target_date:
                continue
            elif post_date < target_date:
                print(f"  → {page}페이지에서 {post_date} 이전 글 발견 → 수집 완료")
                stop = True
                break
            qs     = parse_qs(urlparse(a_tag["href"]).query)
            nid    = qs.get("nid", [None])[0]
            writer = tr.select_one("td.p11")
            writer = writer.get_text(strip=True) if writer else ""
            views  = tds[3].get_text(strip=True) if len(tds) > 3 else ""
            title  = a_tag.get("title") or a_tag.get_text(strip=True)
            posts.append({"date": date_text, "title": title,
                          "writer": writer, "views": views, "nid": nid})
            page_count += 1
        print(f"  {page}페이지: {target_date_str} 글 {page_count}개 (누적 {len(posts)}개)")
        time.sleep(delay)
    return posts

def fetch_body(nid):
    try:
        res  = requests.get(f"{FRONT_API}/discussion/detail",
                            headers=HEADERS_API, params={"id": nid}, timeout=15)
        data = res.json()
        if not data.get("isSuccess"):
            return ""
        html = data["result"].get("contentHtml", "") or ""
        text = BeautifulSoup(html, "html.parser").get_text(separator=" ", strip=True)
        return re.sub(r"\s+", " ", text).strip()
    except Exception:
        return ""

print("✅ 함수 정의 완료")

In [ ]:
posts = collect_posts_for_date(
    code=STOCK_CODE,
    target_date_str=TARGET_DATE,
    delay=DELAY_SEC,
)
print(f"\n총 {len(posts)}개 게시글 발견")

if not posts:
    print("\n[안내] 게시글을 찾지 못했습니다. TARGET_DATE 날짜를 확인하세요.")
else:
    print("\n본문 수집 중…")
    for p in tqdm(posts, desc="본문 수집"):
        p["content"] = fetch_body(p["nid"])
        time.sleep(DELAY_SEC)
    df = pd.DataFrame(posts, columns=["date","title","writer","views","content","nid"])
    filled = (df["content"].str.len() > 0).sum()
    print(f"본문 수집 성공: {filled} / {len(df)}개")
    print(df[["date","title","writer","views"]].head().to_string(index=False))

In [ ]:
if "df" not in dir() or len(df) == 0:
    print("먼저 게시글 수집(Cell 5)을 완료하세요.")
else:
    print(f"감성분석 시작 ({len(df)}개 게시글)…")
    results = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="감성분석"):
        text   = str(row["title"]) + " " + str(row.get("content", ""))
        result = analyze_sentiment(text)
        results.append(result)

    df["sentiment"]        = [r["sentiment"] for r in results]
    df["sentiment_score"]  = [r["score"]     for r in results]
    df["sentiment_reason"] = [r["reason"]    for r in results]

    counts = df["sentiment"].value_counts()
    total  = len(df)
    print("\n── 감성분석 결과 ──")
    for label in ["긍정","중립","부정"]:
        n = counts.get(label, 0)
        bar = "█" * int(n / total * 20)
        print(f"  {label}: {n:4d}개 ({n/total*100:5.1f}%) {bar}")
    print(f"  평균 감성점수: {df['sentiment_score'].mean():.2f} / 5.00")

In [ ]:
def extract_keywords(texts, top_n=30):
    stopwords = {
        "있다","없다","그리고","하지만","그런데","그래서","때문","이번","정말","진짜",
        "이제","아직","계속","조금","많이","너무","모두","경우","생각","오늘","내일",
        "이거","저거","근데","그냥","하면","해서","에서","으로","에게","이런","저런",
        "어떤","얼마","언제","어디","무슨","이미","또한","다시","삼성","전자","삼성전자",
        "종목","주식","투자","이상","이하","합니다","있습","없습","같습","같은","한다",
        "한국","우리","사람","것들","때는","에는","으로","부터","까지","라고","라서",
    }
    word_counts = Counter()
    for text in texts:
        for w in re.findall(r"[가-힣]{2,8}", str(text)):
            if w not in stopwords:
                word_counts[w] += 1
    return word_counts.most_common(top_n)

def make_hourly_trend(df):
    df2 = df.copy()
    df2["hour"] = df2["date"].str.extract(r"(\d{2}):\d{2}").astype(float)
    hourly = (
        df2.groupby("hour")
           .agg(count=("title","count"), avg_score=("sentiment_score","mean"))
           .reset_index()
    )
    hourly["hour"] = hourly["hour"].astype(int).astype(str).str.zfill(2) + "시"
    return hourly

if "df" not in dir() or "sentiment" not in df.columns:
    print("먼저 감성분석(Cell 6)을 완료하세요.")
else:
    all_texts = (df["title"] + " " + df["content"].fillna("")).tolist()
    keywords  = extract_keywords(all_texts, top_n=30)
    df_kw     = pd.DataFrame(keywords, columns=["키워드","언급횟수"])
    df_hourly = make_hourly_trend(df)

    print("=== 전체 TOP 키워드 ===")
    print(df_kw.head(15).to_string(index=False))
    print("\n=== 시간대별 추이 ===")
    print(df_hourly.to_string(index=False))

In [ ]:
def generate_insight(df, df_kw, df_hourly, stock_code, target_date):
    total  = len(df)
    counts = df["sentiment"].value_counts().to_dict()
    avg_sc = df["sentiment_score"].mean()
    pos_n  = counts.get("긍정", 0)
    neg_n  = counts.get("부정", 0)
    neu_n  = counts.get("중립", 0)
    pos_r  = pos_n / total if total else 0
    neg_r  = neg_n / total if total else 0

    # 시간대 분석
    if len(df_hourly) > 0:
        peak_row = df_hourly.loc[df_hourly["count"].idxmax()]
        peak_h   = peak_row["hour"]
        peak_cnt = int(peak_row["count"])
        try:
            peak_h_int = int(peak_h.replace("시", ""))
        except Exception:
            peak_h_int = -1
        if 9 <= peak_h_int <= 10:
            peak_context = "장 시작 직후로, 전일 뉴스·공시 소화 과정에서 투자자 관심이 집중된 것으로 해석됩니다."
        elif 15 <= peak_h_int <= 16:
            peak_context = "장 마감 전후로, 당일 주가 흐름에 대한 반응과 다음 날 전략 논의가 활발했던 것으로 보입니다."
        elif 0 <= peak_h_int <= 8:
            peak_context = "장 개장 전 시간대로, 해외 시장 동향이나 전날 이슈에 대한 선행 반응이 나타난 것으로 볼 수 있습니다."
        else:
            peak_context = "장 중 시간대로, 당일 주가 변동에 실시간으로 반응한 게시글이 다수 발생한 것으로 추정됩니다."
        top3_hours       = df_hourly.nlargest(3, "count")["hour"].tolist()
        low_score_hours  = df_hourly[df_hourly["avg_score"] < 2.8]["hour"].tolist()
        high_score_hours = df_hourly[df_hourly["avg_score"] >= 3.5]["hour"].tolist()
    else:
        peak_h, peak_cnt, peak_context = "불명", 0, ""
        top3_hours, low_score_hours, high_score_hours = [], [], []

    # 테마 태그 분류
    all_kw = df_kw["키워드"].tolist()
    theme_map = {
        "AI·반도체":  ["AI","반도체","엔비디아","HBM","GPU","파운드리","시스템반도체","칩","나노","공정"],
        "미국·해외":  ["미국","나스닥","달러","환율","해외","뉴욕","연준","Fed","금리"],
        "배당·가치":  ["배당","배당금","가치주","자사주","소각","리츠"],
        "ETF·펀드":   ["ETF","펀드","인덱스","패시브","액티브","운용","NAV"],
        "실적·어닝":  ["실적","어닝","매출","영업이익","흑자","적자","가이던스","컨센서스"],
        "수급·거래":  ["외국인","기관","개인","수급","매수","매도","공매도","순매수"],
        "리스크":     ["금리","인플레","환율","지정학","규제","소송","리콜"],
    }
    detected_themes = {}
    for theme, keywords in theme_map.items():
        matched = [k for k in all_kw if any(kw in k or k in kw for kw in keywords)]
        if matched:
            detected_themes[theme] = matched[:3]

    # 감성 심리 진단
    if avg_sc >= 3.8:
        mood_label = "강한 낙관"
        mood_desc = (
            "투자자 게시글의 " + f"{pos_r*100:.1f}" + "%가 긍정적 표현을 포함하고 있으며, "
            "평균 감성점수 " + f"{avg_sc:.2f}" + "점(5점 만점)으로 종목에 대한 신뢰가 높게 유지되고 있습니다. "
            "이 수준의 낙관 심리는 단기 모멘텀 지속 가능성을 시사하며, "
            "ETF 마케팅 측면에서는 해당 종목 비중을 앞세운 콘텐츠 노출 타이밍으로 적합합니다."
        )
    elif avg_sc >= 3.3:
        mood_label = "온건한 낙관"
        mood_desc = (
            "긍정 게시글이 " + f"{pos_r*100:.1f}" + "%로 우세하지만 부정 의견도 " + f"{neg_r*100:.1f}" + "% 존재합니다. "
            "전반적으로 기대감이 앞서면서도 일부 투자자는 리스크를 인식하고 있는 복합적 심리 상태입니다. "
            "이 구간은 투자자가 확신을 찾고 있는 단계로 해석할 수 있어, "
            "근거 중심의 팩트 콘텐츠(실적 데이터, 포트폴리오 비중 근거 등)가 설득력을 가질 시기입니다."
        )
    elif avg_sc >= 2.8:
        mood_label = "중립적 관망"
        mood_desc = (
            "긍정·부정 의견이 혼재되며 평균 감성점수 " + f"{avg_sc:.2f}" + "점으로 방향성이 불분명한 구간입니다. "
            "총 게시글 " + f"{total:,}" + "개 중 중립 게시글이 " + f"{neu_n:,}" + "개(" + f"{neu_n/total*100:.1f}" + "%)로 많아, "
            "투자자들이 결정을 유보하고 정보를 수집하는 단계로 볼 수 있습니다. "
            "관망 심리 시기에는 비교 콘텐츠나 시나리오 기반 콘텐츠가 효과적입니다."
        )
    elif avg_sc >= 2.3:
        mood_label = "온건한 비관"
        mood_desc = (
            "부정 게시글 비중이 " + f"{neg_r*100:.1f}" + "%로 긍정(" + f"{pos_r*100:.1f}" + "%)을 상회합니다. "
            "투자자들이 리스크 요인에 민감하게 반응하고 있으며 단기 하방 압력이 감지됩니다. "
            "이 국면에서 공격적 매수 유도 마케팅은 역효과를 낼 수 있으므로, "
            "장기 관점의 안정성·분산 효과를 강조하는 방어적 메시지가 적합합니다."
        )
    else:
        mood_label = "강한 비관"
        mood_desc = (
            "부정 게시글이 " + f"{neg_r*100:.1f}" + "%로 투자자 불안감이 뚜렷하게 나타납니다. "
            "평균 감성점수 " + f"{avg_sc:.2f}" + "점은 최근 악재나 주가 급락에 따른 심리 위축을 반영합니다. "
            "이 시기 마케팅 메시지는 손실 공포를 자극하지 않는 방향으로 설계되어야 하며, "
            "저점 분할매수·리밸런싱 기회라는 프레이밍이 일부 투자자에게 유효할 수 있습니다."
        )

    top15_kw = df_kw["키워드"].head(15).tolist()

    # 대표 게시글 예시
    pos_df = df[df["sentiment"] == "긍정"][["title", "sentiment_reason"]].head(5)
    neg_df = df[df["sentiment"] == "부정"][["title", "sentiment_reason"]].head(5)
    sep    = "\n"
    pos_ex = sep.join("      · " + r["title"] + " (" + r["sentiment_reason"] + ")"
                      for _, r in pos_df.iterrows())
    neg_ex = sep.join("      · " + r["title"] + " (" + r["sentiment_reason"] + ")"
                      for _, r in neg_df.iterrows())

    # 리스크 레벨
    if neg_r >= 0.5:
        risk_level  = "높음"
        risk_action = (
            "부정 게시글이 절반을 넘어 투자자 신뢰 회복이 최우선 과제입니다. "
            "공식 채널을 통한 팩트체크 콘텐츠, IR 자료 배포, "
            "운용사 공식 입장 제시 등이 단기 대응책으로 권고됩니다."
        )
    elif neg_r >= 0.35:
        risk_level  = "중간"
        risk_action = (
            "부정 의견이 상당수 존재하나 아직 다수 의견은 아닙니다. "
            "핵심 우려 키워드를 모니터링하며 선제적 커뮤니케이션을 준비하는 것이 적절합니다. "
            "부정 게시글에서 반복 등장하는 키워드는 잠재적 FAQ 항목으로 관리할 필요가 있습니다."
        )
    else:
        risk_level  = "낮음"
        risk_action = (
            "현재 부정 여론이 소수로 관리 가능한 수준입니다. "
            "긍정 모멘텀 유지를 위해 투자자 관심 키워드 기반 콘텐츠를 지속 생산하고, "
            "자연 발생하는 긍정 언급을 SNS·광고 소재로 활용하는 전략을 검토할 수 있습니다."
        )

    # 마케팅 실행 시사점
    mkt_actions = []
    if "AI·반도체" in detected_themes:
        kw_str = ", ".join(detected_themes["AI·반도체"])
        mkt_actions.append(
            "【AI·반도체 테마 활용】 투자자 관심에서 AI·반도체 관련 키워드(" + kw_str + ")가 상위에 포착됩니다. "
            "ETF 편입 종목 중 해당 테마 비중을 전면에 내세운 콘텐츠가 높은 반응률을 기대할 수 있습니다. "
            "단순 테마 홍보에 그치지 않고 '왜 이 ETF가 해당 테마를 효율적으로 담는가'에 대한 "
            "구체적 근거(편입 종목 리스트, 비중, 리밸런싱 주기)를 함께 제시해야 설득력이 높아집니다."
        )
    if "배당·가치" in detected_themes:
        kw_str = ", ".join(detected_themes["배당·가치"])
        mkt_actions.append(
            "【배당·가치 수요 대응】 배당·가치 관련 키워드(" + kw_str + ")가 포착되어 "
            "안정적 현금흐름을 원하는 투자자 수요가 존재함을 시사합니다. "
            "배당 수익률 시뮬레이션 콘텐츠(월 배당 환산, 복리 시나리오 등)를 제공하면 "
            "실수요층의 클릭 전환율을 높일 수 있습니다."
        )
    if "미국·해외" in detected_themes:
        kw_str = ", ".join(detected_themes["미국·해외"])
        mkt_actions.append(
            "【해외·미국 ETF 관심 포착】 미국·해외 관련 키워드(" + kw_str + ")가 빈번히 언급됩니다. "
            "환노출 vs 환헤지 구조의 차이, 달러 강세·약세 국면별 성과 비교 등 "
            "투자자가 실질적으로 궁금해하는 정보를 콘텐츠화하면 자연 공유가 기대됩니다. "
            "해외 ETF 가입 장벽을 낮추는 FAQ형 콘텐츠도 유효한 접근입니다."
        )
    if "수급·거래" in detected_themes:
        kw_str = ", ".join(detected_themes["수급·거래"])
        mkt_actions.append(
            "【수급 관심 대응】 외국인·기관 수급 관련 키워드(" + kw_str + ")가 포착됩니다. "
            "수급 데이터는 개인 투자자가 시장 방향성을 판단하는 주요 지표이므로, "
            "ETF 순자산 유입·유출 동향을 정기적으로 공개하고 이를 신뢰 지표로 마케팅에 활용할 수 있습니다."
        )
    if not mkt_actions:
        kw_str = ", ".join(top15_kw[:5])
        mkt_actions.append(
            "【기본 마케팅 방향】 상위 키워드(" + kw_str + " 등)를 중심으로 "
            "투자자 관심사와 접점이 높은 콘텐츠를 기획하세요. "
            "종토방은 실시간 투자 심리를 반영하는 1차 채널로, "
            "반복 등장 키워드는 광고 카피·검색 키워드·SNS 해시태그 기획의 출발점으로 활용도가 높습니다."
        )
    mkt_actions.append(
        "【피크 타임 채널 전략】 게시글이 가장 많이 올라오는 시간대는 " + peak_h + "(" + str(peak_cnt) + "건)이며, "
        + peak_context + " "
        "푸시 알림, 리타겟팅 광고, SNS 포스팅 등 즉각적 반응이 필요한 채널은 이 시간대에 집중 집행하고, "
        "특히 " + ", ".join(top3_hours[:2]) + " 시간대에 콘텐츠 노출을 집중하는 운영 스케줄을 권고합니다."
    )

    # 종합 의견
    theme_names = ", ".join(list(detected_themes.keys())[:2]) if detected_themes else "주요 테마"
    if avg_sc >= 3.5 and neg_r < 0.3:
        conclusion = (
            "  " + target_date + " 기준 " + stock_code + " 종목에 대한 투자자 심리는 전반적으로 건강한 낙관 구간에 있습니다. "
            "상위 키워드에서 포착된 테마(" + theme_names + ") 관련 콘텐츠를 " + peak_h + " 전후로 집중 배포하고, "
            "긍정 모멘텀이 유지되는 이 시기를 신규 투자자 유입 캠페인의 적기로 활용하는 것을 권고합니다. "
            "다만 부정 의견(" + f"{neg_r*100:.1f}" + "%)이 완전히 소멸하지 않은 만큼, "
            "핵심 우려 키워드에 대한 선제적 FAQ 콘텐츠를 병행하여 신뢰도를 공고히 하는 투 트랙 전략이 적합합니다."
        )
    elif avg_sc <= 2.5 or neg_r >= 0.45:
        conclusion = (
            "  " + target_date + " 기준 " + stock_code + " 종목의 투자자 심리는 경계가 필요한 구간입니다. "
            "부정 게시글 비중 " + f"{neg_r*100:.1f}" + "%는 단순 불만 수준을 넘어 투자자 신뢰 훼손 가능성을 내포합니다. "
            "이 시기 마케팅 커뮤니케이션의 핵심은 매수 유도가 아닌 신뢰 회복에 있으며, "
            "운용사 공식 입장·운용 전략의 일관성·장기 성과 데이터를 전면에 내세운 "
            "방어적·설명적 콘텐츠 전략이 현 시점에서 가장 적합한 대응입니다."
        )
    else:
        conclusion = (
            "  " + target_date + " 기준 " + stock_code + " 종목은 긍정·부정 의견이 혼재하는 전환점에 위치합니다. "
            "투자자들이 방향성을 탐색 중인 이 구간은 정보 민감도가 높아 "
            "양질의 팩트 콘텐츠가 의사결정에 직접적인 영향을 미칠 수 있습니다. "
            "상위 키워드(" + ", ".join(top15_kw[:3]) + ")를 중심으로 투자자 질문에 선제적으로 답하는 "
            "콘텐츠를 피크 시간대(" + peak_h + ")에 배포하고, "
            "감성 취약 시간대의 부정 심리를 완화할 수 있는 데이터 기반 안심 메시지를 병행하는 이중 전략을 권고합니다."
        )

    # 리포트 조립
    div  = "=" * 65
    div2 = "-" * 65
    L    = []
    L += [div,
          "  종합 마케팅 인사이트 리포트",
          "  종목: " + stock_code + "  |  분석 날짜: " + target_date + "  |  총 게시글: " + f"{total:,}" + "개",
          div, ""]
    L += ["【1. 투자자 심리 종합 진단】",
          "  현재 심리 상태: " + mood_label + " (평균 감성점수 " + f"{avg_sc:.2f}" + "/5.00)",
          "", "  " + mood_desc, ""]
    L += ["【2. 감성 분포 상세】",
          "  긍정 " + f"{pos_n:,}" + "건(" + f"{pos_r*100:.1f}" + "%)  |  "
          + "중립 " + f"{neu_n:,}" + "건(" + f"{neu_n/total*100:.1f}" + "%)  |  "
          + "부정 " + f"{neg_n:,}" + "건(" + f"{neg_r*100:.1f}" + "%)",
          "",
          "  긍정 의견 주요 사례 (감지 근거 포함):",
          pos_ex if pos_ex else "      · 해당 없음",
          "",
          "  부정 의견 주요 사례 (감지 근거 포함):",
          neg_ex if neg_ex else "      · 해당 없음",
          ""]
    L += ["【3. 핵심 관심 키워드 TOP 15】",
          "  " + " / ".join(top15_kw),
          "",
          "  위 키워드는 해당 날짜 투자자들이 가장 많이 언급한 주제어입니다. "
          "반복 등장 키워드일수록 투자자 관심이 높은 이슈이므로, "
          "콘텐츠 기획·광고 카피·검색 키워드 설정 시 1순위로 검토하세요.", ""]
    if detected_themes:
        L.append("【4. 감지된 테마 태그】")
        for theme, matched_kw in detected_themes.items():
            L.append("  · " + theme + ": " + ", ".join(matched_kw))
        L += ["",
              "  테마 태그는 투자자가 이 종목을 어떤 맥락에서 소비하는지를 보여줍니다. "
              "마케팅 메시지 설계 시 감지된 테마를 중심으로 USP(핵심 차별점)를 구성하면 "
              "타겟 투자자의 공감을 더 효과적으로 이끌어낼 수 있습니다.", ""]
    L += ["【5. 게시글 활동 패턴 및 채널 전략】",
          "  피크 시간대: " + peak_h + " (" + f"{peak_cnt:,}" + "건)",
          "  활동 상위 3개 시간대: " + ", ".join(top3_hours)]
    if high_score_hours:
        L.append("  감성 양호 시간대(3.5점↑): " + ", ".join(high_score_hours) + " → 긍정 콘텐츠 집중 배포 권고")
    if low_score_hours:
        L.append("  감성 취약 시간대(2.8점↓): " + ", ".join(low_score_hours) + " → 불안 심리 완화 콘텐츠 필요")
    L += ["", "  " + peak_context, ""]
    L += ["【6. 리스크 평가 및 커뮤니케이션 대응】",
          "  리스크 레벨: " + risk_level, "", "  " + risk_action, ""]
    L.append("【7. 마케팅 실행 시사점】")
    L.append("")
    for action in mkt_actions:
        L.append("  " + action)
        L.append("")
    L += ["【8. 종합 의견】", "", conclusion, "",
          div2,
          "※ 본 리포트는 종목토론방 게시글 텍스트 감성분석 결과이며,",
          "   투자 권유 또는 투자 조언이 아닙니다.",
          "   마케팅 전략 수립 시 추가적인 정량 데이터와 함께 종합 검토하세요.",
          div2]
    return "\n".join(L)


if "df_kw" not in dir():
    print("먼저 키워드 분석(Cell 7)을 완료하세요.")
else:
    insight_text = generate_insight(df, df_kw, df_hourly, STOCK_CODE, TARGET_DATE)
    print(insight_text)


In [ ]:
# ── ETF 데이터 파일 업로드 (선택 사항) ───────────────────────────────
# 두 파일 모두 선택 사항입니다.
# 업로드하지 않으면 ETF 채널 분석 없이 종토방 인사이트만 실행됩니다.
#
# ① ETF 순매수 데이터 (엑셀)  : 은행·증권사 채널별 수급 분석에 활용
# ② ETF 종목 분석 리포트 (PDF): AUM·수익률·보수·변동성 인사이트에 활용

ETF_FILE_PATH = ""   # 엑셀 파일 경로
ETF_PDF_PATH  = ""   # PDF 파일 경로
etf_result    = None
pdf_etf_data  = {}

try:
    from google.colab import files as colab_files

    print("─" * 50)
    print("① ETF 순매수 데이터 (엑셀) 업로드")
    print("   취소하면 ETF 채널 수급 분석을 건너뜁니다.")
    print("─" * 50)
    up1 = colab_files.upload()
    if up1:
        ETF_FILE_PATH = list(up1.keys())[0]
        print(f"✅ 엑셀 업로드 완료: {ETF_FILE_PATH}\n")
    else:
        print("엑셀 업로드 없음 → ETF 채널 수급 분석 생략\n")

    print("─" * 50)
    print("② ETF 종목 분석 리포트 (PDF) 업로드")
    print("   취소하면 AUM·수익률·보수·변동성 인사이트를 건너뜁니다.")
    print("─" * 50)
    up2 = colab_files.upload()
    if up2:
        ETF_PDF_PATH = list(up2.keys())[0]
        print(f"✅ PDF 업로드 완료: {ETF_PDF_PATH}\n")
    else:
        print("PDF 업로드 없음 → ETF 종목 리포트 인사이트 생략\n")

except Exception as e:
    print(f"파일 업로드 불가 ({e})")
    print("ETF_FILE_PATH / ETF_PDF_PATH 를 직접 입력하세요.")
    print('  예) ETF_FILE_PATH = "ETF_순매수_데이터.xlsx"')
    print('      ETF_PDF_PATH  = "ETF_리포트.pdf"')

In [ ]:

# ── ETF 순매수 데이터 분석 ─────────────────────────────────────────────
# Cell 9 없이 단독 실행해도 오류 없도록 변수 기본값 설정
if 'ETF_FILE_PATH' not in dir(): ETF_FILE_PATH = ""
if 'ETF_PDF_PATH'  not in dir(): ETF_PDF_PATH  = ""
if 'etf_result'    not in dir(): etf_result    = None
if 'pdf_etf_data'  not in dir(): pdf_etf_data  = {}

from collections import defaultdict
import re as _re

LEVERAGE_KEYWORDS = [
    "레버리지","인버스","선물","2X","3X","곱버스","커버드콜",
    "데일리","단일종목","합성","스왑","파생","futures","F 2"
]

def classify_etf_type(name):
    for kw in LEVERAGE_KEYWORDS:
        if kw in name:
            return "leverage"
    return "equity"

def parse_pdf_etf(pdf_path):
    """
    ETF 비교 리포트 PDF 파싱.
    반환: {code_6자리: {name, aum_억, type, total_fee,
                        ret_1m/3m/6m/1y, volatility_1y, tracking_error, category}}
    """
    try:
        import pdfplumber
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "pdfplumber", "-q"])
        import pdfplumber

    with pdfplumber.open(pdf_path) as pdf:
        full = "\n".join(p.extract_text() or "" for p in pdf.pages)

    # ── 종목코드: 헤더 줄에서 | 069500 패턴으로 추출 ──────────────────
    header_lines = [l for l in full.split('\n')
                    if ('시장대표' in l or '레버리지/인버스' in l or '업종섹터' in l)
                    and _re.search(r'\|\s*\d{6}', l)]
    if not header_lines:
        return {}
    header_line = header_lines[0]

    # 카테고리 순서 파악 (코드 없는 레버리지 포함)
    cat_order = _re.findall(
        r'((?:국내주식|해외주식|레버리지/인버스|채권|원자재)[^\|]*)',
        header_line
    )
    codes_from_header = _re.findall(r'\|\s*(\d{6})\b', header_line)

    # ── 종목명: ETF사 브랜드명으로 분리 ──────────────────────────────
    brand_pat = r'(?:KODEX|TIGER|KBSTAR|HANARO|SOL|RISE|PLUS|ACE|ARIRANG)'
    name_line_m = _re.search(
        rf'({brand_pat}[^\n]+)', full
    )
    if not name_line_m:
        return {}
    name_line = name_line_m.group(1)
    # 다음 줄 suffix 확인 ("레버리지" 등이 잘려 붙는 경우)
    nl_end = name_line_m.end()
    next_line = full[nl_end:full.find('\n', nl_end+1)].strip() \
                if nl_end < len(full) else ""

    raw_names = _re.split(rf'(?={brand_pat}\s)', name_line.strip())
    etf_names = [n.strip() for n in raw_names if n.strip()]

    # suffix 처리: 다음 줄이 브랜드 아닌 한글이면 마지막 단일종목 ETF에 붙임
    if next_line and not _re.match(rf'(?:{brand_pat}|수익률|기준|구성|위험|운용)', next_line):
        for i, nm in enumerate(etf_names):
            if '단일종목' in nm and '레버리지' not in nm:
                etf_names[i] = nm + ' ' + next_line
                break

    # ── 수치 파싱 ──────────────────────────────────────────────────────
    def parse_ret_line(line, label):
        """'1개월 23.73 1개월 48.01 1개월 1개월 10.49 1개월 15.58'
           → [23.73, 48.01, None, 10.49, 15.58]  (수익률 없는 ETF는 None)"""
        tokens = _re.split(rf'({label})', line)
        result = []
        for i, tok in enumerate(tokens):
            if tok == label:
                nxt = tokens[i+1].strip() if i+1 < len(tokens) else ""
                num = _re.match(r'^([\d.]+)', nxt)
                result.append(float(num.group(1)) if num else None)
        return result

    ret_block_m = _re.search(r'기간별 수익률\(%\)(.*?)구성종목', full, _re.S)
    ret1m = ret3m = ret1y = [None] * len(etf_names)
    if ret_block_m:
        block = ret_block_m.group(1)
        for line in block.strip().split('\n'):
            label = line.split()[0] if line.strip() else ''
            if label == '1개월': ret1m = parse_ret_line(line, '1개월')
            if label == '3개월': ret3m = parse_ret_line(line, '3개월')
            if label == '1년':   ret1y = parse_ret_line(line, '1년')

    # AUM (페이지2에서 순서대로)
    aum_vals = [int(v.replace(",",""))
                for v in _re.findall(r'순자산\(억원\)\s*([\d,]+)', full)]

    # 총보수 (앞 N개만)
    fee_matches = list(_re.finditer(r'총보수\(%\)\s+([\d.]+)', full))
    fee_vals = [float(m.group(1)) for m in fee_matches[:len(etf_names)]]

    # 변동성·추적오차율: 반복 헤더 줄 다음 숫자 줄에서 5개
    def parse_multi_col(label, text):
        pat = label + r'(?:\s+' + label + r')+\n([\d.\- ]+)'
        m = _re.search(pat, text)
        if not m:
            return []
        return [float(v) if v.strip() != '-' else None
                for v in m.group(1).split()]

    vol_vals = parse_multi_col('변동성', full)
    te_vals  = parse_multi_col('추적오차율', full)

    # ── 매핑: ETF명 순서 = AUM/보수/수익률 순서 ──────────────────────
    # 코드는 헤더에서 3개만 나오므로, 이름 순서와 대조해 매핑
    # 이름 순서: [KODEX200, KODEX레버리지, KODEX삼성전자단일종목레버리지,
    #             TIGER미국S&P500, TIGER반도체TOP10]
    # 코드 순서: [069500, 360750, 396500] (코드 있는 것만)
    # → 이름-코드 매핑은 이름에서 코드 포함 여부로 결정하거나,
    #   카테고리 순서가 헤더 순서와 같으므로 index로 매핑
    # 헤더에 코드 없는 ETF: 레버리지/인버스 2종 → 코드를 파일명에서 추출 시도
    import os
    fname = os.path.basename(pdf_path)
    # 파일명: "ETF-KR7069500007_KR7122630007_KR70193W0003_KR7360750004_KR7396500001.pdf"
    fname_codes = _re.findall(r'KR7(\w{6})', fname)  # ['069500','122630','0193W0','360750','396500']

    # ETF명 개수와 파일명 코드 개수가 맞으면 순서대로 매핑
    use_codes = fname_codes if len(fname_codes) == len(etf_names) else \
                (codes_from_header + ['unknown'] * (len(etf_names)-len(codes_from_header)))

    result = {}
    for i, (code, name) in enumerate(zip(use_codes, etf_names)):
        result[code] = {
            'name':          name,
            'type':          classify_etf_type(name),
            'aum_억':        aum_vals[i]   if i < len(aum_vals)   else None,
            'total_fee':     fee_vals[i]   if i < len(fee_vals)   else None,
            'ret_1m':        ret1m[i]      if i < len(ret1m)      else None,
            'ret_3m':        ret3m[i]      if i < len(ret3m)      else None,
            'ret_1y':        ret1y[i]      if i < len(ret1y)      else None,
            'volatility_1y': vol_vals[i]  if i < len(vol_vals)   else None,
            'tracking_error':te_vals[i]   if i < len(te_vals)    else None,
        }
    return result

def load_etf_data(filepath):
    import openpyxl
    wb     = openpyxl.load_workbook(filepath, read_only=True)
    sheets = [s for s in wb.sheetnames if s != '참고사항']
    def to_int(v):
        if v is None or v == '-': return 0
        try: return int(v)
        except: return 0
    data = defaultdict(dict)
    for sheet_name in sheets:
        ws = wb[sheet_name]
        for i, row in enumerate(ws.iter_rows(values_only=True)):
            if i == 0: continue
            code, name = row[0], row[1]
            if not code or code == '단축코드' or not name or name == '전체': continue
            data[code][sheet_name] = {
                'name':        name,
                'institution': to_int(row[2]),
                'foreign':     to_int(row[3]),
                'individual':  to_int(row[4]),
                'fin_invest':  to_int(row[6]),
                'bank':        to_int(row[10]),
                'etf_type':    classify_etf_type(name),
            }
    return data, sheets

def analyze_etf(etf_data, sheets, target_stock_code, target_date_str, pdf_etf_data=None):
    from datetime import datetime

    # AUM 맵: PDF 코드(6자리) → AUM(억)
    aum_map = {}
    if pdf_etf_data:
        for code, d in pdf_etf_data.items():
            if d.get('aum_억'):
                aum_map[code[:6]] = d['aum_억']

    def get_code_short(full_code):
        return full_code.split('*')[0][:6]

    STOCK_NAME_MAP = {
        '005930': ['삼성전자','반도체','AI','MSCI Korea','KRX','200','KOSPI'],
        '000660': ['SK하이닉스','반도체','AI','HBM'],
        '035420': ['NAVER','인터넷','IT'],
        '005380': ['현대차','자동차'],
        '051910': ['LG화학','2차전지','배터리'],
        '373220': ['LG에너지솔루션','2차전지','배터리'],
    }
    related_keywords = STOCK_NAME_MAP.get(target_stock_code, [target_stock_code])
    related_etfs = {
        code: list(w.values())[0]['name']
        for code, w in etf_data.items()
        if any(kw in list(w.values())[0]['name'] for kw in related_keywords)
    }

    # 기준 주차
    try:
        target_dt = datetime.strptime(target_date_str, "%Y.%m.%d")
        tm, td    = target_dt.month, target_dt.day
    except:
        tm, td = 0, 0

    target_week = sheets[-1]
    for sheet in sheets:
        try:
            parts = sheet.split('-')
            sm, sd = int(parts[0].split('.')[0]), int(parts[0].split('.')[1])
            em, ed = int(parts[1].split('.')[0]), int(parts[1].split('.')[1])
            if (sm, sd) <= (tm, td) <= (em, ed):
                target_week = sheet
                break
        except:
            continue

    # 채널별 TOP5 (유형 구분 + AUM 강도)
    def build_ranking(channel_key, top_n=5):
        totals = {}
        for code, weeks in etf_data.items():
            total  = sum(w.get(channel_key, 0) for w in weeks.values())
            name   = list(weeks.values())[0]['name']
            etype  = list(weeks.values())[0].get('etf_type', classify_etf_type(name))
            cshort = get_code_short(code)
            aum    = aum_map.get(cshort)
            inten  = (total / (aum * 1e8) * 100) if (aum and aum > 0) else None
            totals[code] = {'name': name, 'total': total,
                            'intensity': inten, 'aum': aum, 'type': etype}
        eq  = sorted([(c,v) for c,v in totals.items()
                      if v['type']=='equity'   and v['total']>0],
                     key=lambda x: x[1]['total'], reverse=True)[:top_n]
        lev = sorted([(c,v) for c,v in totals.items()
                      if v['type']=='leverage' and v['total']>0],
                     key=lambda x: x[1]['total'], reverse=True)[:top_n]
        return eq, lev

    bank_eq_top,  bank_lev_top  = build_ranking('bank')
    fin_eq_top,   fin_lev_top   = build_ranking('fin_invest')

    week_bank = {s: sum(w[s]['bank']      for w in etf_data.values() if s in w) for s in sheets}
    week_fin  = {s: sum(w[s]['fin_invest'] for w in etf_data.values() if s in w) for s in sheets}
    cur_bank  = week_bank.get(target_week, 0)
    cur_fin   = week_fin.get(target_week,  0)
    prev_idx  = sheets.index(target_week) - 1
    prev_bank = week_bank.get(sheets[prev_idx], 0) if prev_idx >= 0 else 0
    prev_fin  = week_fin.get(sheets[prev_idx],  0) if prev_idx >= 0 else 0

    rel_week, rel_bank_t, rel_fin_t, rel_ind_t = [], {}, {}, {}
    for code, name in related_etfs.items():
        if code in etf_data:
            rel_bank_t[code] = sum(w.get('bank',0)       for w in etf_data[code].values())
            rel_fin_t[code]  = sum(w.get('fin_invest',0)  for w in etf_data[code].values())
            rel_ind_t[code]  = sum(w.get('individual',0)  for w in etf_data[code].values())
            if target_week in etf_data[code]:
                d = etf_data[code][target_week]
                rel_week.append({'name': name,
                                  'bank': d.get('bank',0),
                                  'fin_invest': d.get('fin_invest',0),
                                  'individual': d.get('individual',0)})
    rel_week.sort(key=lambda x: abs(x['bank']), reverse=True)

    return {
        'target_week':   target_week,
        'related_etfs':  related_etfs,
        'bank_eq_top':   bank_eq_top,  'bank_lev_top': bank_lev_top,
        'fin_eq_top':    fin_eq_top,   'fin_lev_top':  fin_lev_top,
        'cur_bank':      cur_bank,     'cur_fin':      cur_fin,
        'bank_trend':    "증가" if cur_bank > prev_bank else "감소",
        'fin_trend':     "증가" if cur_fin  > prev_fin  else "감소",
        'rel_week_data': rel_week,
        'rel_bank_total':rel_bank_t,   'rel_fin_total': rel_fin_t,
        'rel_ind_total': rel_ind_t,
        'week_bank':     week_bank,    'sheets':        sheets,
        'pdf_etf_data':  pdf_etf_data or {},
    }

# ── 실행 ──────────────────────────────────────────────────────────────
etf_result   = None
pdf_etf_data = {}

if ETF_PDF_PATH:
    try:
        pdf_etf_data = parse_pdf_etf(ETF_PDF_PATH)
        print(f"✅ PDF 파싱 완료: {len(pdf_etf_data)}개 ETF 정보 추출")
        for code, d in pdf_etf_data.items():
            aum_str = f"{d['aum_억']:,}억" if d.get('aum_억') else "?"
            print(f"   {code} | {d['name']} | AUM {aum_str} | {d['type']} | 1개월 {d.get('ret_1m')}%")
    except Exception as e:
        print(f"⚠️ PDF 파싱 오류: {e}")
        import traceback; traceback.print_exc()

if ETF_FILE_PATH:
    etf_data, etf_sheets = load_etf_data(ETF_FILE_PATH)
    etf_result = analyze_etf(etf_data, etf_sheets, STOCK_CODE, TARGET_DATE, pdf_etf_data)
    print(f"\n✅ ETF 분석 완료 | 기준 주차: {etf_result['target_week']} | 관련 ETF: {len(etf_result['related_etfs'])}개")
    if pdf_etf_data:
        # AUM 매핑 결과 확인
        aum_hits = sum(1 for c,v in etf_result['bank_eq_top']
                       if v.get('intensity') is not None)
        print(f"   AUM 매핑 성공: {aum_hits}/{len(etf_result['bank_eq_top'])}개 (은행 순수주식 TOP5 기준)")
else:
    print("ETF 엑셀 미업로드 → ETF 채널 분석 생략")


In [ ]:

def generate_etf_insight(etf_result, stock_code, target_date):
    if not etf_result:
        return ""

    r           = etf_result
    target_week = r['target_week']
    cur_bank    = r['cur_bank']
    cur_fin     = r['cur_fin']
    bank_trend  = r['bank_trend']
    fin_trend   = r['fin_trend']
    bank_eq_top = r['bank_eq_top']
    bank_lev_top= r['bank_lev_top']
    fin_eq_top  = r['fin_eq_top']
    fin_lev_top = r['fin_lev_top']
    rel_etfs    = r['related_etfs']
    rel_week    = r['rel_week_data']
    rel_bank_t  = r['rel_bank_total']
    rel_fin_t   = r['rel_fin_total']
    rel_ind_t   = r['rel_ind_total']
    pdf_data    = r.get('pdf_etf_data', {})

    def fmt(v):
        val  = abs(v) / 1e8
        sign = "+" if v >= 0 else "-"
        return f"{sign}{val:.1f}억"

    def intensity_str(item):
        """순매수 금액 + AUM 있을 때만 강도 표시"""
        d     = item[1]
        name  = d['name']
        total = d['total']
        inten = d['intensity']
        aum   = d['aum']
        base  = f"{name[:24]} {fmt(total)}"
        if inten is not None:
            level = "★★★ 매우강" if inten >= 50 else \
                    "★★  강"     if inten >= 20 else \
                    "★   보통"   if inten >= 5  else \
                         "미미"
            return f"{base} | AUM대비 {inten:.1f}% [{level}]"
        elif aum:
            return f"{base} | AUM {aum:,}억"
        else:
            return base

    div  = "=" * 65
    div2 = "-" * 65
    L    = []

    L += [div,
          "  ETF 채널 마케팅 인사이트 (순매수 데이터 연계 분석)",
          "  종목: " + stock_code + "  |  기준 주차: " + target_week,
          div, ""]

    # ── 섹션 A: 채널 전체 흐름 ────────────────────────────────────────
    L.append("【A. 마케팅 채널별 시장 전체 흐름】")
    L.append("")
    bank_dir = "순매수" if cur_bank >= 0 else "순매도"
    fin_dir  = "순매수" if cur_fin  >= 0 else "순매도"
    L.append(
        "  " + target_week + " 주차 기준, 은행 채널 ETF 전체 " + bank_dir + " 규모는 "
        + fmt(cur_bank) + "으로 전주 대비 " + bank_trend + "했습니다. "
        "은행 채널 순매수는 신탁·퇴직연금·자체 운용 자금을 포함하므로, "
        "은행 창구 및 앱을 통한 ETF 프로모션 효과의 간접 지표로 활용할 수 있습니다."
    )
    L.append("")
    L.append(
        "  금융투자(증권사) 채널 전체 " + fin_dir + " 규모는 " + fmt(cur_fin)
        + "으로 전주 대비 " + fin_trend + "했습니다. "
        "금융투자 순매수에는 LP 헤지 물량이 포함되어 있어 개인 순매수가 증가하면 "
        "반대로 움직이는 경향이 있으므로, 증권사 마케팅 효과 측정 시 "
        "개인 순매수 데이터와 반드시 교차 검증이 필요합니다."
    )
    L.append("")

    # ── 섹션 B: 채널별 TOP5 (유형 구분 + AUM 강도) ───────────────────
    L.append("【B. 채널별 누적 순매수 TOP 5 — 유형 구분 및 AUM 대비 유입강도】")
    L.append("")
    L.append("  ※ AUM 대비 유입강도 = 전체 기간 누적 순매수 ÷ AUM × 100(%)")
    L.append("     ★★★ 매우강(50%↑)  ★★ 강(20%↑)  ★ 보통(5%↑)  미미(5%↓)")
    L.append("     AUM 정보는 PDF 리포트 업로드 시 자동 반영됩니다.")
    L.append("")

    for channel, eq_top, lev_top, label in [
        ("은행", bank_eq_top, bank_lev_top, "은행 채널"),
        ("증권사", fin_eq_top, fin_lev_top, "금융투자(증권사) 채널"),
    ]:
        L.append("  ▶ " + label + " — 순수 주식형 ETF TOP 5")
        if eq_top:
            for i, item in enumerate(eq_top, 1):
                L.append(f"    {i}. {intensity_str(item)}")
        else:
            L.append("    (해당 없음)")
        L.append("")

        L.append("  ▶ " + label + " — 레버리지·파생·인버스형 ETF TOP 5")
        if lev_top:
            for i, item in enumerate(lev_top, 1):
                L.append(f"    {i}. {intensity_str(item)}")
            L.append("")
            L.append(
                "  레버리지·인버스형은 AUM 규모가 작아도 단기 방향성 베팅으로 유입이 집중될 수 있습니다. "
                "AUM 대비 유입강도가 높은 종목은 단기 시장 심리 방향성을 가늠하는 선행 지표로 활용하세요. "
                "반면 순수 주식형에서 AUM 대비 유입강도가 높은 종목은 기관·채널의 중장기 선호 신호로 해석됩니다."
            )
        else:
            L.append("    (해당 없음)")
        L.append("")

    # ── 섹션 C: PDF 리포트 연계 인사이트 ─────────────────────────────
    if pdf_data:
        L.append("【C. ETF 리포트 연계 인사이트】")
        L.append("")

        # PDF 종목 중 채널 TOP에 등장한 것 교차 분석
        # bank eq top에서 PDF AUM 있는 것 찾기
        def find_pdf_match(top_list):
            matches = []
            for code, d in top_list:
                cshort = code.split('*')[0][:6]
                if cshort in pdf_data:
                    matches.append((cshort, d, pdf_data[cshort]))
            return matches

        bank_eq_pdf  = find_pdf_match(bank_eq_top)
        fin_eq_pdf   = find_pdf_match(fin_eq_top)
        bank_lev_pdf = find_pdf_match(bank_lev_top)

        if bank_eq_pdf or fin_eq_pdf or bank_lev_pdf:
            L.append("  ▶ 채널 상위 종목 × PDF 리포트 교차 분석")
            L.append("")

        for cshort, flow_d, pdf_d in (bank_eq_pdf + fin_eq_pdf)[:4]:
            name      = pdf_d.get('name', flow_d['name'])
            aum       = pdf_d.get('aum_억')
            fee       = pdf_d.get('total_fee')
            ret1m     = pdf_d.get('ret_1m')
            ret1y     = pdf_d.get('ret_1y')
            vol       = pdf_d.get('volatility_1y')
            te        = pdf_d.get('tracking_error')
            inten     = flow_d.get('intensity')
            total_flow= flow_d.get('total', 0)

            L.append(f"  [{name}]")

            # 수익률 해석
            if ret1m is not None and ret1y is not None:
                if ret1m > 15:
                    ret_comment = (
                        f"최근 1개월 수익률 {ret1m}%로 단기 강세가 두드러집니다. "
                        f"1년 수익률 {ret1y}%의 장기 성과와 함께 마케팅 소재로 활용 가능합니다."
                    )
                elif ret1m > 0:
                    ret_comment = (
                        f"최근 1개월 수익률 {ret1m}%, 1년 {ret1y}%로 꾸준한 성과를 보이고 있습니다. "
                        f"단기 급등보다 안정적 성장 스토리 중심의 메시지가 적합합니다."
                    )
                else:
                    ret_comment = (
                        f"최근 1개월 수익률 {ret1m}%로 단기 조정 구간에 있습니다. "
                        f"수익률 부각보다 분산투자·장기 보유 관점의 메시지가 방어적으로 유효합니다."
                    )
                L.append(f"  · 수익률: {ret_comment}")

            # AUM + 유입강도 해석
            if aum and inten is not None:
                if inten >= 20:
                    flow_comment = (
                        f"AUM {aum:,}억 대비 유입강도 {inten:.1f}%로 채널 자금 유입이 매우 두드러집니다. "
                        f"이 수준의 유입은 채널 마케팅 효과가 실제 수급으로 연결되고 있다는 "
                        f"긍정적 신호로 해석할 수 있습니다."
                    )
                elif inten >= 5:
                    flow_comment = (
                        f"AUM {aum:,}억 대비 유입강도 {inten:.1f}%로 의미 있는 채널 유입이 확인됩니다. "
                        f"프로모션 지속 시 유입 강도가 더 높아질 여지가 있습니다."
                    )
                else:
                    flow_comment = (
                        f"AUM {aum:,}억 대비 유입강도 {inten:.1f}%로 아직 유입 규모는 소규모입니다. "
                        f"채널별 추천 상품 탑재 여부를 재검토할 필요가 있습니다."
                    )
                L.append(f"  · 유입강도: {flow_comment}")

            # 보수 해석
            if fee is not None:
                if fee <= 0.10:
                    fee_comment = (
                        f"총보수 {fee}%로 동일 유형 대비 매우 낮은 수준입니다. "
                        f"장기 보유 투자자에게 비용 효율성을 강조하는 마케팅이 차별화 포인트가 됩니다."
                    )
                elif fee <= 0.30:
                    fee_comment = (
                        f"총보수 {fee}%로 적정 수준입니다. "
                        f"보수 외 추적오차·거래 편의성 등 비용 이외의 가치를 함께 제시하는 것이 효과적입니다."
                    )
                else:
                    fee_comment = (
                        f"총보수 {fee}%로 상대적으로 높은 편입니다. "
                        f"높은 보수를 정당화할 수 있는 액티브 수익률·테마 프리미엄·접근성 등 "
                        f"차별화 근거를 마케팅 메시지에 포함해야 합니다."
                    )
                L.append(f"  · 보수: {fee_comment}")

            # 변동성 해석
            if vol is not None:
                if vol >= 50:
                    vol_comment = (
                        f"1년 변동성 {vol}%로 매우 높은 위험 자산입니다. "
                        f"적극 투자 성향 고객 전용 채널(MTS 투자 탭, 증권사 테마 추천)에 집중 배치하고, "
                        f"투자 경고 문구를 함께 노출하는 방식이 규제 리스크도 줄입니다."
                    )
                elif vol >= 25:
                    vol_comment = (
                        f"1년 변동성 {vol}%로 중위험 수준입니다. "
                        f"은행 퇴직연금·신탁 채널에서도 소화 가능한 위험도이므로 "
                        f"채널 다변화 전략이 유효합니다."
                    )
                else:
                    vol_comment = (
                        f"1년 변동성 {vol}%로 낮은 편입니다. "
                        f"안정 추구 고객층이 많은 은행 채널 퇴직연금 라인업 편입을 적극 검토할 수 있습니다."
                    )
                L.append(f"  · 변동성: {vol_comment}")
            L.append("")

        # PDF에 있는 레버리지형 채널 상위 종목
        if bank_lev_pdf:
            L.append("  ▶ 레버리지형 채널 상위 종목 리포트 연계")
            for cshort, flow_d, pdf_d in bank_lev_pdf[:2]:
                name  = pdf_d.get('name', flow_d['name'])
                vol   = pdf_d.get('volatility_1y')
                ret1m = pdf_d.get('ret_1m')
                L.append(f"  [{name}]")
                if ret1m and vol:
                    L.append(
                        f"  · 1개월 수익률 {ret1m}%, 변동성 {vol}%로 고위험·고수익 구간에 있습니다. "
                        "레버리지 상품은 단기 방향성 베팅 수요를 반영하므로, "
                        "종토방 긍정 심리 피크 시간대와 레버리지 유입 강도를 교차 모니터링하면 "
                        "시장 과열 여부를 사전에 감지하는 지표로 활용할 수 있습니다."
                    )
                L.append("")

    elif not pdf_data:
        L.append("【C. ETF 리포트 연계 인사이트】")
        L.append("")
        L.append(
            "  ETF 비교 리포트 PDF가 업로드되지 않아 수익률·보수·변동성 기반 심층 분석을 생략합니다. "
            "PDF를 함께 업로드하면 채널 TOP 종목별 AUM 자동 반영, "
            "수익률·보수·변동성 기반 마케팅 메시지 자동 생성 기능이 활성화됩니다."
        )
        L.append("")

    # ── 섹션 D: 관련 ETF 채널 분석 ───────────────────────────────────
    if rel_etfs:
        L.append("【D. " + stock_code + " 관련 ETF 채널 수급 분석】")
        L.append(f"  분석 대상 관련 ETF: {len(rel_etfs)}개")
        L.append("")
        if rel_week:
            L.append("  ▶ " + target_week + " 주차 채널별 수급 현황")
            for d in rel_week[:5]:
                name  = d['name']
                bank  = d['bank']
                fin   = d['fin_invest']
                indiv = d['individual']
                L.append(
                    f"    · {name[:26]}"
                    f" | 은행 {'↑' if bank>=0 else '↓'}{fmt(bank)}"
                    f" | 증권사 {'↑' if fin>=0 else '↓'}{fmt(fin)}"
                    f" | 개인 {'↑' if indiv>=0 else '↓'}{fmt(indiv)}"
                )
            L.append("")

        total_rb = sum(rel_bank_t.values())
        total_rf = sum(rel_fin_t.values())
        total_ri = sum(rel_ind_t.values())
        L += ["  ▶ 전체 기간 관련 ETF 채널 합계",
              f"    은행 채널:   {fmt(total_rb)}",
              f"    증권사 채널: {fmt(total_rf)}",
              f"    개인 투자자: {fmt(total_ri)}", ""]

        if total_rb > 0 and total_ri > 0:
            L.append(
                "  은행 채널과 개인이 동반 순매수 중입니다. 기관과 리테일 수요가 "
                "일치하는 국면으로 채널 마케팅 집행 효율이 높은 시기입니다."
            )
        elif total_rb > 0 and total_ri <= 0:
            L.append(
                "  은행은 순매수하지만 개인은 관망·매도 중입니다. "
                "은행 채널 중심 기관 수요는 있으나 리테일 확산이 아직 이루어지지 않은 단계로, "
                "개인 투자자 대상 인지도 제고 콘텐츠를 병행하면 시너지를 기대할 수 있습니다."
            )
        elif total_rb <= 0 and total_ri > 0:
            L.append(
                "  개인 순매수는 활발하나 은행 채널은 소극적입니다. "
                "리테일 수요가 먼저 형성된 상황이므로 은행 RM 채널에 "
                "해당 ETF 라인업 편입을 제안하기 적합한 타이밍입니다."
            )
        else:
            L.append(
                "  현재 관련 ETF에 대한 채널·개인 모두 순매도 우위입니다. "
                "수비적 커뮤니케이션(보유 유지 메시지, 장기 성과 데이터 제공)에 집중하세요."
            )
        L.append("")

    # ── 섹션 E: 채널 마케팅 실행 권고 ────────────────────────────────
    L.append("【E. 채널 마케팅 실행 권고】")
    L.append("")
    L.append(
        "  【은행 채널 전략】 순수 주식형 ETF에서 AUM 대비 유입강도가 높은 종목은 "
        "은행 RM 추천 상품 목록에 이미 편입되어 효과가 나타나고 있을 가능성이 높습니다. "
        "이 종목들을 중심으로 고객 설명 자료(팩트시트)를 주기적으로 업데이트하고, "
        "유입강도가 갑자기 떨어지는 주차를 감지하면 RM 교육 및 캠페인을 재점검하세요."
    )
    L.append("")
    L.append(
        "  【증권사 채널 전략】 레버리지형 ETF의 AUM 대비 유입강도를 모니터링해 "
        "단기 방향성 베팅이 집중되는 시기를 포착하면 관련 테마 ETF의 MTS 노출 강화 시점을 "
        "선제적으로 결정할 수 있습니다. LP 차감 후 실질 순매수를 추정하려면 "
        "개인 순매수와 역방향 구간의 금융투자 물량을 LP 효과로 간주해 제거하세요."
    )
    L.append("")
    L.append(
        "  【주간 리뷰 체계】 ETF 순매수 데이터는 주 단위 집계이므로 매주 금요일 업데이트 후 "
        "다음 주 채널 마케팅 방향을 조정하는 Weekly Review를 운영하는 것을 권고합니다. "
        "종토방 일별 감성 피크 시간대와 주간 채널 수급 흐름을 함께 트래킹하면 "
        "커뮤니티 심리 변화가 실제 채널 수급으로 전환되는 시차(lag)를 파악할 수 있습니다."
    )
    L.append("")
    L += [div2,
          "※ ETF 순매수는 마케팅 효과의 간접 지표이며 직접 인과관계를 입증하지 않습니다.",
          "   캠페인 집행 내역·RM 피드백 등 정성 데이터와 함께 해석하세요.",
          div2]
    return "\n".join(L)


if "df_kw" not in dir():
    print("먼저 키워드 분석(Cell 7)을 완료하세요.")
elif etf_result is None:
    etf_insight_text = ""
    print("ETF 파일 미업로드 → ETF 채널 인사이트 생략")
else:
    etf_insight_text = generate_etf_insight(etf_result, STOCK_CODE, TARGET_DATE)
    print(etf_insight_text)


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment

def save_excel_full(df, df_kw, df_hourly, insight_text,
                    etf_insight_text, stock_code, target_date):

    def hdr(ws, row, cols, hex_color="2F5597"):
        fill = PatternFill("solid", fgColor=hex_color)
        for c in range(1, cols + 1):
            cell = ws.cell(row=row, column=c)
            cell.fill = fill
            cell.font = Font(bold=True, color="FFFFFF")
            cell.alignment = Alignment(horizontal="center", vertical="center")

    wb = Workbook()

    # 시트1: 게시글 전체
    ws1 = wb.active
    ws1.title = "게시글 전체"
    h1 = ["날짜","제목","작성자","조회수","감성","감성점수","감성근거","본문","nid"]
    ws1.append(h1)
    hdr(ws1, 1, len(h1))
    s_color = {"긍정":"C6EFCE","부정":"FFC7CE","중립":"FFEB9C"}
    for _, r in df.iterrows():
        ws1.append([
            r.get("date",""), r.get("title",""), r.get("writer",""),
            r.get("views",""), r.get("sentiment",""), r.get("sentiment_score",""),
            r.get("sentiment_reason",""), str(r.get("content",""))[:500], r.get("nid","")
        ])
        ws1.cell(ws1.max_row, 5).fill = PatternFill("solid",
            fgColor=s_color.get(r.get("sentiment","중립"), "FFFFFF"))
    for col, w in [("A",18),("B",40),("C",16),("D",10),("E",10),
                   ("F",10),("G",35),("H",60),("I",14)]:
        ws1.column_dimensions[col].width = w
    ws1.freeze_panes = "A2"
    ws1.auto_filter.ref = ws1.dimensions

    # 시트2: 감성 분석
    ws2 = wb.create_sheet("감성 분석")
    total  = len(df)
    counts = df["sentiment"].value_counts()
    ws2.append(["감성 분석 요약"])
    ws2["A1"].font = Font(bold=True, size=14, color="2F5597")
    ws2.append([])
    for label, val in [("종목코드",stock_code),("분석 날짜",target_date),
                       ("총 게시글",total),
                       ("평균 감성점수",round(df["sentiment_score"].mean(),2))]:
        ws2.append([label, val])
        ws2.cell(ws2.max_row, 1).font = Font(bold=True)
    ws2.append([])
    ws2.append(["감성","게시글 수","비율(%)"])
    hdr(ws2, ws2.max_row, 3)
    for label, hex_c in [("긍정","70AD47"),("중립","FFC000"),("부정","FF0000")]:
        n = counts.get(label, 0)
        ws2.append([label, n, round(n/total*100, 1)])
        ws2.cell(ws2.max_row, 1).fill = PatternFill("solid", fgColor=hex_c)
        ws2.cell(ws2.max_row, 1).font = Font(bold=True, color="FFFFFF")
    for col, w in [("A",14),("B",14),("C",12)]:
        ws2.column_dimensions[col].width = w

    # 시트3: 키워드 분석
    ws3 = wb.create_sheet("키워드 분석")
    ws3.append(["키워드 분석"])
    ws3["A1"].font = Font(bold=True, size=14, color="2F5597")
    ws3.append([])
    ws3.append(["순위","키워드","언급횟수"])
    hdr(ws3, ws3.max_row, 3)
    for i, row in df_kw.iterrows():
        ws3.append([i+1, row["키워드"], row["언급횟수"]])
    for col, w in [("A",8),("B",16),("C",14)]:
        ws3.column_dimensions[col].width = w

    # 시트4: 시간대 추이
    ws4 = wb.create_sheet("시간대 추이")
    ws4.append(["시간대 추이"])
    ws4["A1"].font = Font(bold=True, size=14, color="2F5597")
    ws4.append([])
    ws4.append(["시간대","게시글 수","평균 감성점수"])
    hdr(ws4, ws4.max_row, 3)
    for _, row in df_hourly.iterrows():
        ws4.append([row["hour"], int(row["count"]), round(row["avg_score"],2)])
    for col, w in [("A",12),("B",14),("C",16)]:
        ws4.column_dimensions[col].width = w

    # 시트5: 종토방 인사이트
    ws5 = wb.create_sheet("종토방 인사이트")
    ws5.append([f"종합 마케팅 인사이트 — {stock_code} ({target_date})"])
    ws5["A1"].font = Font(bold=True, size=14, color="2F5597")
    ws5.append([])
    for line in insight_text.split("\n"):
        ws5.append([line])
        cell = ws5.cell(ws5.max_row, 1)
        cell.alignment = Alignment(wrap_text=True)
        stripped = line.strip()
        if stripped and len(stripped) > 1 and stripped[:2] in ["【1","【2","【3","【4","【5","【6","【7","【8"]:
            cell.font = Font(bold=True)
    ws5.column_dimensions["A"].width = 110

    # 시트6: ETF 채널 인사이트 (있을 때만)
    if etf_insight_text:
        ws6 = wb.create_sheet("ETF 채널 인사이트")
        ws6.append([f"ETF 채널 마케팅 인사이트 — {stock_code}"])
        ws6["A1"].font = Font(bold=True, size=14, color="1F4E79")
        ws6.append([])
        for line in etf_insight_text.split("\n"):
            ws6.append([line])
            cell = ws6.cell(ws6.max_row, 1)
            cell.alignment = Alignment(wrap_text=True)
            stripped = line.strip()
            if stripped and len(stripped) > 1 and stripped[:2] in ["【A","【B","【C","【D"]:
                cell.font = Font(bold=True)
        ws6.column_dimensions["A"].width = 110

    out = f"naver_jongtobang_{stock_code}_{target_date.replace('.','')}_통합분석.xlsx"
    wb.save(out)
    return out


# ── 실행 ──────────────────────────────────────────────────────────────
required = ["df","df_kw","df_hourly","insight_text"]
missing  = [v for v in required if v not in dir()]
if missing:
    print(f"먼저 완료해야 할 셀: {missing}")
else:
    etf_txt = etf_insight_text if "etf_insight_text" in dir() else ""
    print("엑셀 저장 중…")
    out = save_excel_full(df, df_kw, df_hourly, insight_text,
                          etf_txt, STOCK_CODE, TARGET_DATE)
    sheets_info = "게시글 전체 / 감성 분석 / 키워드 분석 / 시간대 추이 / 종토방 인사이트"
    if etf_txt:
        sheets_info += " / ETF 채널 인사이트"
    print(f"\n✅ 저장 완료: {out}")
    print(f"   시트: {sheets_info}")
    try:
        from google.colab import files
        files.download(out)
    except Exception:
        print("왼쪽 파일 탐색기에서 직접 다운로드하세요.")